### 07 · Rent Proxy
 Computes assessed value per sqft as a proxy for commercial rent.
 Uses NYC PLUTO `assesstot` / `lotarea` matched via BallTree.

**Input:** `csv/00_base_data.csv`, PLUTO CSV
**Output:** `csv/07_rent_proxy.csv` — `osm_id`, `assess_per_sqft`

In [2]:
# ── Cell 2 · Imports ──────────────────────────────────
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree

df = pd.read_csv("csv/00_base_data.csv")
print(f"pandas {pd.__version__}")
print(f"Loaded {len(df)} records")

PLUTO_CSV = "../ramy/NYC_pluto_25v4_csv/pluto_25v4.csv"

pandas 3.0.2
Loaded 2915 records


In [3]:
# ── Cell 3 · Load PLUTO & Compute Rent Proxy ──────────

print("Loading PLUTO data...")
pluto = pd.read_csv(PLUTO_CSV, low_memory=False)
pluto_mn = pluto[pluto["borough"] == "MN"].dropna(subset=["latitude", "longitude"]).copy()
print(f"  {len(pluto_mn)} Manhattan lots loaded")

# Compute assessed value per sqft
pluto_mn["assess_per_sqft"] = pluto_mn["assesstot"] / pluto_mn["lotarea"].replace(0, np.nan)

# Build spatial index
pluto_coords = np.radians(pluto_mn[["latitude", "longitude"]].values)
tree = BallTree(pluto_coords, metric="haversine")

# Match each shop to nearest lot
shop_coords = np.radians(df[["lat", "lon"]].values)
distances, indices = tree.query(shop_coords, k=1)

df["assess_per_sqft"] = pluto_mn.iloc[indices.flatten()]["assess_per_sqft"].values

print(f"\nassess_per_sqft fill : {df['assess_per_sqft'].notna().sum()}/{len(df)}")
print(f"  Mean : {df['assess_per_sqft'].mean():.2f}")
print(f"  Min  : {df['assess_per_sqft'].min():.2f}")
print(f"  Max  : {df['assess_per_sqft'].max():.2f}")

Loading PLUTO data...
  42116 Manhattan lots loaded

assess_per_sqft fill : 2915/2915
  Mean : 2261.57
  Min  : 0.00
  Max  : 29245.86


In [4]:
# ── Cell 4 · Save ─────────────────────────────────────
df_out = df[["osm_id", "assess_per_sqft"]]
df_out.to_csv("csv/07_rent_proxy.csv", index=False, encoding="utf-8")
print(f"Saved {len(df_out)} records to csv/07_rent_proxy.csv")
print(df_out.describe().round(2))

Saved 2915 records to csv/07_rent_proxy.csv
             osm_id  assess_per_sqft
count  2.915000e+03          2915.00
mean   6.217806e+09          2261.57
std    4.166958e+09          2228.70
min    8.539400e+07             0.00
25%    2.709934e+09           834.16
50%    5.726058e+09          1505.69
75%    1.023030e+10          3026.57
max    1.380607e+10         29245.86
